In [2]:
# dotenv initialization
import dotenv
import os

dotenv.load_dotenv()

True

In [3]:
import pandas as pd
from langchain_core.documents import Document

data = pd.read_csv("dataset/Tamil_movies_dataset.csv")
df = pd.read_csv("dataset/tamil.csv") 

def generate_unified_profile(row):
    # Mapping logic for different headers
    name = row.get('MovieName') or row.get('Title')
    genre = row.get('Genre')
    director = row.get('Director')
    actor = row.get('Actor') or row.get('Cast')
    year = row.get('Year') or row.get('Release Year')
    rating = row.get('Rating')
    plot = row.get('Plot', 'No plot available')

    return (
        f"Title: {name}\n"
        f"Genre: {genre}\n"
        f"Director: {director}\n"
        f"Actor: {actor}\n"
        f"Release Year: {year}\n"
        f"Rating: {rating}\n"
        f"Synopsis: {plot}\n"
    )

data["profile"] = data.apply(generate_unified_profile, axis=1)
docs_1 = [
    Document(
        page_content=row["profile"],
        metadata={
            "source": "Tamil_movies_dataset",
            "title": row.get('MovieName'),
            "genre": row.get('Genre'),
            "year": row.get('Year')
        }
    ) for _, row in data.iterrows()
]

df["profile"] = df.apply(generate_unified_profile, axis=1)
docs_2 = [
    Document(
        page_content=row["profile"],
        metadata={
            "source": "Tamil_dataset_2",
            "title": row.get('Title'),
            "genre": row.get('Genre'),
            "year": row.get('Release Year')
        }
    ) for _, row in df.iterrows()
]

documents = docs_1 + docs_2

print(f"Total documents prepared for TrailerCraft: {len(documents)}")


Total documents prepared for TrailerCraft: 745


In [4]:
# Embeddings 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(documents,embeddings,persist_directory="dataset/Tamil_movies_dataset_chroma")

In [5]:
# Output Classes
from pydantic import BaseModel,Field
from typing import List

class DirectorConsultation(BaseModel):
    style_tip: str = Field(description="A tip on how to adapt the director's signature filmmaking style.")
    trademark_interval: str = Field(description="A description of a signature interval block typical for this director's style.")

class Shot(BaseModel):
    shotnumber:int
    visual:str=Field(description="A brief description of the visual content of the shot.")
    cameraangle:str=Field(description="The camera angle used in the shot, e.g., close-up, wide shot, aerial view.")
    audio_cue:str=Field(description="Any significant audio cues present in the shot, such as dialogue, sound effects, or music.")

class TrailerPackage(BaseModel):
    structure: str = Field(description="The 3-act breakdown of the trailer")
    voice_over: str = Field(description="The script for the narrator it should be in tamil and that tamil is not pure it should be a tamil in a way that it is used in common")
    music_mood: str = Field(description="Instrumentation, tempo, and vibe")
    fonrstyle: str = Field(description="The font style to be used in the trailer")
    title: str = Field(description="The title of the movie in tamil and english that is good make it catchy and appealing")
    shot_list: List[Shot]
    director_consultation: DirectorConsultation


In [6]:
# Prompt template for the model
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

retrivar = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt =ChatPromptTemplate.from_messages([
    ("system",    """You are a Kollywood Trailer Editor who is an expert in creating engaging and captivating trailers for Tamil movies.
    Your task is to analyze the provided movie plot and generate a detailed trailer structure that includes a 3-act breakdown, voice-over script, music mood, and a shot list with descriptions of visuals, camera angles, and audio cues using the synopsis of the movie also use the conservation hsitory as well"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", """
    CONTEEXT FROM DATABASE: {context}
    MOVIE SYNOPSIS: {synopsis}
    ADDITIONAL STYLE GUIDELINES: {instructions}
    """)
])

In [ ]:
# LLM Integration with Groq and Gemini
import json
import re
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_classic.memory import ChatMessageHistory
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),temperature=0.7)
chat_history=ChatMessageHistory()

def segregate_intent(user_input, history):
    parser_prompt = f"""
    Analyze the user request: "{user_input}"
    1. If the user asks for a specific thing (like JUST a title), set 'mode' to 'specific'.
    2. Otherwise, set 'mode' to 'full'.
    3. Extract the plot into 'synopsis'.
    4. Extract director style into 'style'.
    5. Extract any other requests into 'instructions'.

    Return ONLY JSON: {{
        "synopsis": "...", 
        "mode": "full/specific", 
        "target": "title/bgm/script/none",
        "style": "...",
        "instructions": "..."
    }}
    """
    raw_response = model.invoke(parser_prompt).content
    clean_json = re.sub(r"```json|```", "", raw_response).strip()
    return json.loads(clean_json)

def generate_trailer_package(user_input: str) -> TrailerPackage:
    parsed = segregate_intent(user_input, chat_history.messages)
    relevant_docs = retrivar.invoke(parsed['synopsis'])
    context_data = "\n\n".join([doc.page_content for doc in relevant_docs])
    if parsed['mode'] == 'specific':
        specific_prompt = f"Context: {context_data}\nHistory: {chat_history.messages}\nUser: {user_input}\nAnswer ONLY the specific request briefly."
        response = model.invoke(specific_prompt).content
        print(response)
        chat_history.add_user_message(user_input)
        chat_history.add_ai_message(response)
        return response
    else:
        structeredllm = model.with_structured_output(TrailerPackage)
        chain = prompt | structeredllm
        response = chain.invoke({
            "chat_history": chat_history.messages,
            "context": context_data,
            "synopsis": parsed['synopsis'],
            "instructions": f"Style: {parsed.get('style', 'General')}. Extra: {parsed.get('instructions', 'None')}"
        })
        
        _print_formatted_output(response)
        
        chat_history.add_user_message(user_input)
        chat_history.add_ai_message(f"Generated trailer: {response.title}")
                
        return response

def _print_formatted_output(response):
    data = response.model_dump()
    for key, value in data.items():
        if key == "director_consultation": continue
        print(f"\n{key.replace('_', ' ').upper()}")
        if isinstance(value, list):
            i=1
            for item in value:
                print(f"Shot {item.get('shot_number', i)}:\n{item.get('visual')}")
                print(f"Camera Angle: {item.get('cameraangle')}")
                print(f"Audio Cue: {item.get('audio_cue')}")
                i+=1
        else:
            print(value)
    
    print("\nDIRECTOR CONSULTATION")
    print(f"Tip: {response.director_consultation.style_tip}")
    print(f"Interval: {response.director_consultation.trademark_interval}")

In [12]:
response=generate_trailer_package("I want a trailer for a Tamil romantic comedy with a twist of mystery, set in Chennai. The trailer should have a vibrant and energetic vibe, with a mix of humor and suspense. The title should be catchy and appealing in both Tamil and English.")


STRUCTURE
Act 1: The Dream and The Detour - Introduces Harish, a struggling filmmaker with big dreams, embarking on a journey to Singapore. It highlights his initial excitement followed by the immediate misfortunes and the loss of his passport, setting up his predicament. Act 2: The Unlikely Alliance and The Unexpected Encounter - Focuses on Harish meeting the eccentric cameraman Vaanambaadi, their comedic partnership, and the near-realization of Harish's dream. This act takes a turn with the introduction of the girl battling cancer, shifting the tone towards a blend of humor and seriousness. Act 3: The Madcap Journey of Life and Death - Showcases the evolving relationship between Harish and the girl, the hilarious yet poignant situations they face, and the overall philosophical undertone of life, death, and everything in between, culminating in a montage of emotional and comedic highs.

VOICE OVER
எங்கேயோ ஒரு மூலையில, ஒரு சின்ன ஃபிலிம் மேக்கர்... பெரிய கனவோட சிங்கப்பூருக்கு கிளம்புறா

In [13]:
repsonse = generate_trailer_package("Just give me a catchy title in Tamil and English for a romantic comedy with a twist of mystery, set in Chennai.")

**Tamil:** சென்னை காதல் புதிர்
**English:** Chennai Love Code
